# HW3 — Binary-entropy bracket verifier from scratch — **Theorem 1**

**Reading.** [`PAPER-ARXIV.md`](../../../../PAPER-ARXIV.md) **§3.2 Theorem 1** and **Appendix A** (the proof). See also the standalone walk-through in [`proof_walk.ipynb`](proof_walk.ipynb) — pure markdown + numeric checks of each algebraic step.

**Goal.** Build `hbin_inverse`, `upper_HR`, `lower_Fano`, a random verifier of $\mathrm{lower}(H) \le \varepsilon \le \mathrm{upper}(H)$, grid-search $(\varepsilon^*, w^*) = (1/5, 0.1610)$, and (**Q5, new in r2.1**) explicitly construct the two extremal families of **Proposition 3.5** (no closed-form improvement of the sandwich): an HR-saturating sequence that drives $\varepsilon \to h/2$ and a Fano-saturating sequence that drives $\varepsilon \to H_\mathrm{bin}^{-1}(h)$.

**Julia companion (optional).** [`julia-theory/notebooks/05_bracket_envelope.jl`](../../../julia-theory/notebooks/05_bracket_envelope.jl) is the reactive slider twin of Q2; [`06_uniform_slack.jl`](../../../julia-theory/notebooks/06_uniform_slack.jl) derives $\varepsilon^*=1/5$ symbolically via `Symbolics.jl` — read alongside Q4.

## Setup

Resolve `onboarding/projects/` on `sys.path`, attach the reflection log, and import the canonical references (used only for spot-checks; the student version derives its own implementations).

In [ ]:
import os, sys
from pathlib import Path
_here = Path(os.getcwd()).resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'shared' / '__init__.py').exists() and _p.name == 'projects':
        _PROJECTS = _p
        break
else:
    raise RuntimeError('could not locate onboarding/projects/')
_REPO = _PROJECTS.parent.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless for CI
import matplotlib.pyplot as plt

from onboarding.projects import reflect
reflect.start(notebook='hw3', store=_PROJECTS / '.reflection.jsonl')
print(f'[setup] projects root: {_PROJECTS}')


In [ ]:
from math import log2
def hbin(p: float) -> float:
    return 0.0 if (p <= 0 or p >= 1) else -(p*log2(p) + (1-p)*log2(1-p))


## Q1 — `hbin_inverse(h)` via bisection on $[0, 1/2]$.

$H_\mathrm{bin}$ is strictly increasing on $[0, 1/2]$, so its inverse on that branch is well-defined.

In [ ]:
def hbin_inverse(h: float, tol: float = 1e-12) -> float:
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish.** Closed-form inverse is impossible (transcendental); we use bisection.

In [ ]:
for eps in [0.05, 0.1, 0.2, 0.3, 0.4, 0.5]:
    print(f'  eps={eps:.2f} -> H={hbin(eps):.6f} -> H^{{-1}}(H)={hbin_inverse(hbin(eps)):.6f}')


**Gate Q1.** Round-trip error $< 10^{-9}$ on the grid.

In [ ]:
for eps in [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]:
    rt = hbin_inverse(hbin(eps))
    assert abs(rt - eps) < 1e-9, f'Q1: round-trip fails at eps={eps}: {rt}'
print('[GATE OK] Q1: hbin_inverse round-trips to 1e-9 on six grid points')


In [ ]:
reflect.log('Q3.Q1_hbin_inverse', 'bisection inverse round-trips ≤1e-9', 'HIGH')


## Q2 — `upper_HR(h) = h/2` (Hellman–Raviv) and `lower_Fano(h) = H^{-1}(h)` (Fano), envelope plot.

**Concept (Theorem 1).** For binary $Y$ and any finite partition $\Pi$,

$$
H_\mathrm{bin}^{-1}\!\big(H(Y\mid\Pi)\big) \;\le\; \varepsilon^*_\Pi \;\le\; \tfrac{1}{2}\, H(Y\mid\Pi).
$$

**Naming the endpoints (Appendix A).**

- **Upper = Hellman–Raviv per cell + Jensen.** For each cell $C$, $e_C \le \tfrac{1}{2} H_\mathrm{bin}(e_C)$ (Hellman–Raviv 1970). Multiply by $q_C$ and sum: $\varepsilon^* = \sum_C q_C e_C \le \tfrac{1}{2}\sum_C q_C H_\mathrm{bin}(e_C) = \tfrac{1}{2} H(Y\mid\Pi)$. That is `upper_HR(h) = h/2` — no Jensen needed, just linearity.
- **Lower = Fano per cell + concavity inversion.** Fano (1961) for binary $Y$ in cell $C$: $H_\mathrm{bin}(e_C) \le H(Y\mid C)$ (here equality by construction). Apply $H_\mathrm{bin}^{-1}$ on the $[0, 1/2]$ branch (monotone), then aggregate via concavity of $H_\mathrm{bin}^{-1} \circ H_\mathrm{bin}$ — the **Jensen** step that gives `lower_Fano(h) = H_bin^{-1}(h)` is the only non-trivial inequality in the proof.

The bracket has width 0 at $h=0$ and at $h=1$; widest near $h \approx H_\mathrm{bin}(1/5) \approx 0.7219$ (Q4).

In [ ]:
def upper_HR(h: float) -> float:
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish — *which* inequality goes which way.** A common error is to swap them. The mnemonic: **HR upper-bounds error by entropy/2** (cheap to compute), **Fano lower-bounds error by $H^{-1}$** (expensive — requires bisection). Their gap is $w(h) = h/2 - H_\mathrm{bin}^{-1}(h)$, peaking at $w^* \approx 0.1610$.

In [ ]:
hs = np.linspace(0, 1, 401)
ups = np.array([upper(h) for h in hs])
los = np.array([lower(h) for h in hs])
fig, ax = plt.subplots(figsize=(6, 4))
ax.fill_between(hs, los, ups, color='C2', alpha=0.25, label='slack region')
ax.plot(hs, ups, label='upper = h/2', lw=2)
ax.plot(hs, los, label='lower = $H_{bin}^{-1}(h)$', lw=2)
ax.set_xlabel('h = H(Y|Π)'); ax.set_ylabel('ε'); ax.legend()
ax.set_title('Q2 — bracket envelope and slack')
plt.tight_layout()
_plots = _PROJECTS / 'psets' / 'hw3' / 'plots'; _plots.mkdir(exist_ok=True)
fig.savefig(_plots / 'hw3_q2_envelope.png', dpi=120); plt.show()


**Gate Q2.** lower ≤ upper everywhere; widest near $h \approx 0.72$ (corresponding to $\varepsilon^* = 1/5$).

In [ ]:
assert np.all(los <= ups + 1e-12), 'Q2: lower > upper somewhere — bug'
gap = ups - los
h_argmax = hs[int(np.argmax(gap))]
assert 0.65 < h_argmax < 0.78, f'Q2: argmax h={h_argmax} not in expected band near 0.72'
assert gap.max() > 0.10, f'Q2: max gap={gap.max()} too small'
print(f'[GATE OK] Q2: lower ≤ upper; max gap {gap.max():.4f} at h≈{h_argmax:.3f}')


In [ ]:
reflect.log('Q3.Q2_bracket', 'lower ≤ upper; widest near h≈0.72', 'HIGH')


## Q3 — Random-sample verifier of $\mathrm{lower}(H) \le \varepsilon \le \mathrm{upper}(H)$.

Sample cell masses via Multinomial; sample per-cell errors uniformly in $[0, 1/2]$ (Bayes errors live there).

In [ ]:
def random_partition_stats(rng, m: int, n: int):
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish.** Sampling errors $> 1/2$ would falsely violate; clamping to $[0, 1/2]$ is essential.

In [ ]:
ok = verify_bracket(num_samples=2000)
print(f'verify_bracket(2000) -> {ok}')


**Gate Q3.** `verify_bracket` returns True on 2000 samples.

In [ ]:
assert ok is True, 'Q3: verifier did not return True'
print('[GATE OK] Q3: 2000 random partitions all in bracket')


In [ ]:
reflect.log('Q3.Q3_verifier', 'bracket holds on 2000 random multinomial+uniform partitions', 'HIGH')


## Q4 — Grid search $\varepsilon^*, w^*$.

Maximise $w(\varepsilon) = H_\mathrm{bin}(\varepsilon)/2 - \varepsilon$ on $[0, 1/2]$.

In [ ]:
def find_w_star(grid: int = 5000):
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish — false lead.** The slack maximum is NOT at $\varepsilon = 1/2$ (the entropy maximum); first-order condition gives $H_\mathrm{bin}'(\varepsilon) = 2 \Rightarrow \varepsilon^* = 1/5$.

In [ ]:
eps_star, w_star = find_w_star(5000)
print(f'eps* = {eps_star:.4f} (expected 0.2000)')
print(f'w*   = {w_star:.4f} (expected 0.1610)')


**Gate Q4.** $\varepsilon^* \in (0.199, 0.201)$ and $w^* \in (0.160, 0.162)$.

In [ ]:
assert 0.199 < eps_star < 0.201, f'Q4: eps*={eps_star} out of band'
assert 0.160 < w_star  < 0.162, f'Q4: w*={w_star} out of band'
print(f'[GATE OK] Q4: eps*={eps_star:.4f}, w*={w_star:.4f}')


In [ ]:
reflect.log('Q3.Q4_wstar', f'eps*={eps_star:.4f}, w*={w_star:.4f}', 'HIGH')


## Q4.5 — **Proposition 3.5** sharpness: both endpoints are saturated by explicit families.

**Statement (Paper §3.2).** No closed-form improvement of the sandwich exists: for any $h \in (0, 1)$ there are partition families $(\Pi_\alpha^\mathrm{HR})_{\alpha}$ and $(\Pi_\alpha^\mathrm{J})_{\alpha}$ with $H(Y\mid\Pi_\alpha) \to h$ such that $\varepsilon^*(\Pi_\alpha^\mathrm{HR}) \to h/2$ (upper saturated) and $\varepsilon^*(\Pi_\alpha^\mathrm{J}) \to H_\mathrm{bin}^{-1}(h)$ (lower saturated).

**Constructions.**

- **HR-saturating** ($\Pi^\mathrm{HR}$, two-mass): one cell with mass $1-\alpha$ and $e=0$, one cell with mass $\alpha$ and $e=1/2$. Then $\varepsilon = \alpha/2$, $H = \alpha$, so $\varepsilon = H/2$ **exactly** — the upper endpoint is hit with equality.
- **Fano-saturating** ($\Pi^\mathrm{J}$, constant-$e$): every cell has the same $e_C = e$ (any cell sizes). Then $H = H_\mathrm{bin}(e)$ and $\varepsilon = e = H_\mathrm{bin}^{-1}(H)$ **exactly** — the lower endpoint is hit with equality.

In [ ]:
def witness_HR(alpha: float):
    """TODO(student): implement.

    See the §Concept and §Distinguish cells above for the
    derivation. The §Gate cell that follows will fail until
    you replace this body."""
    raise NotImplementedError("TODO(student): see §Concept above")


**Distinguish — sharpness vs tightness.** Sharpness = endpoints are attained by *some* feasible $\Pi$; tightness on a *specific* $\Pi$ would mean *equality* there. Proposition 3.5 is about the envelope, not about any single problem.

In [ ]:
alphas = np.linspace(0.05, 0.95, 19)
hr_eps, hr_H = zip(*[witness_HR(a) for a in alphas])
hr_eps, hr_H = np.array(hr_eps), np.array(hr_H)
hr_residual = hr_eps - hr_H/2
print(f'HR witness: max|ε - H/2| = {np.max(np.abs(hr_residual)):.2e}')

es = np.linspace(0.02, 0.48, 24)
fa_eps, fa_H = zip(*[witness_Fano(e) for e in es])
fa_eps, fa_H = np.array(fa_eps), np.array(fa_H)
fa_residual = fa_eps - np.array([hbin_inverse(h) for h in fa_H])
print(f'Fano witness: max|ε - H_bin^{{-1}}(H)| = {np.max(np.abs(fa_residual)):.2e}')

hs = np.linspace(0, 1, 401)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.fill_between(hs, [lower(h) for h in hs], [upper(h) for h in hs], color='C2', alpha=0.18, label='Theorem 1 bracket')
ax.plot(hr_H, hr_eps, 'o-', color='C3', label='HR-saturating Π^HR (upper)')
ax.plot(fa_H, fa_eps, 's-', color='C0', label='Fano-saturating Π^J (lower)')
ax.set_xlabel('H(Y|Π)'); ax.set_ylabel('ε(Π)'); ax.legend()
ax.set_title('Q4.5 — Proposition 3.5 sharpness witnesses')
plt.tight_layout()
_plots = _PROJECTS / 'psets' / 'hw3' / 'plots'; _plots.mkdir(exist_ok=True)
fig.savefig(_plots / 'hw3_q45_sharpness.png', dpi=120); plt.show()


**Gate Q4.5.** Both residuals $< 10^{-9}$ on their respective grids.

In [ ]:
assert np.max(np.abs(hr_residual)) < 1e-9, f'Q4.5: HR witness off by {np.max(np.abs(hr_residual))}'
assert np.max(np.abs(fa_residual)) < 1e-9, f'Q4.5: Fano witness off by {np.max(np.abs(fa_residual))}'
print(f'[GATE OK] Q4.5: both Proposition 3.5 witnesses saturate the bracket to ≤1e-9')


In [ ]:
reflect.log('Q3.Q4.5_sharpness',
            'Prop 3.5 witnesses constructed: 2-mass family saturates upper, constant-e family saturates lower (residual ≤1e-9)',
            'HIGH')


## Q5 — Writeup & calibration.

In [ ]:
reflect.log('Q3.Q5_writeup', 'four sections + five calibration entries; bracket holds end-to-end', 'MEDIUM')
print('HW3 reflection log:')
for r in reflect.dump():
    if 'hw3' in r['notebook']:
        print(f"  [{r['level']:>10}] {r['concept']}: {r['claim']}")


**End of HW3.**